# 04 — Bayesian PyMC example

Example notebook: conjugate inference by hand, then a **small** hierarchical
logistic in PyMC on a StatsBomb subset. This is **not** the production module.

 MCMC is deliberately tiny (176 shots, 12 players, 4 chains × 500 draws).


## 0. Setup


In [1]:
from __future__ import annotations

import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
from scipy import stats
from scipy.special import expit
from statsmodels.stats.proportion import proportions_ztest

from football_intelligence.features.shots import read_shot_dataset

warnings.filterwarnings("ignore", category=FutureWarning, module="arviz")

SEED = 42
az.style.use("arviz-darkgrid")
plt.rcParams.update({"figure.figsize": (8.2, 3.3), "figure.dpi": 110})
print(f"PyMC {pm.__version__}  ArviZ {az.__version__}")

PyMC 5.28.5  ArviZ 0.23.4


## 1. Bayesian foundations

**Bayes' theorem** (up to the normalising constant we often skip in sampling):

\[
p(\theta \mid y) \;=\; \frac{p(y \mid \theta)\, p(\theta)}{p(y)}
\qquad\Longleftrightarrow\qquad
\underbrace{p(\theta \mid y)}_{\text{posterior}}
\;\propto\;
\underbrace{p(y \mid \theta)}_{\text{likelihood}}
\,\times\,
\underbrace{p(\theta)}_{\text{prior}}
\]

| Term | Meaning |
|---|---|
| **Parameter** \(\theta\) | Unknown we infer (a conversion rate \(p\), a coefficient, a player effect). Not random in nature — *uncertain given the data*. |
| **Prior** \(p(\theta)\) | Belief about \(\theta\) *before* seeing these observations. Encodes information and regularisation. |
| **Likelihood** \(p(y \mid \theta)\) | Sampling model: how the data would look if \(\theta\) were known. |
| **Posterior** \(p(\theta \mid y)\) | Updated belief after the data. The object of inference. |
| **Posterior predictive** \(p(\tilde y \mid y)\) | Distribution of *new data* after integrating the posterior: \(p(\tilde y \mid y)=\int p(\tilde y \mid \theta)\,p(\theta \mid y)\,d\theta\). Uncertainty in \(\theta\) is folded in. |

### Bayesian vs frequentist uncertainty

- **Frequentist**: \(\theta\) is a fixed unknown. Probability statements are about *procedures* under repeated sampling. A 95% confidence interval covers the true \(\theta\) in 95% of repetitions of the *experiment*; it does not say \(P(\theta \in \text{CI} \mid y)=0.95\).
- **Bayesian**: probability is a coherent degree of belief. After seeing \(y\), \(P(\theta \in I \mid y)=0.95\) is a statement about this \(\theta\) given this data (and the prior).

Neither automatically answers a causal question. Both can be wrong if the sampling model is wrong.

### Credible interval vs confidence interval

- **95% credible interval**: an interval of the *posterior*. \(P(\theta \in I \mid y)=0.95\). Common choices: equal-tailed quantiles, or highest posterior density (HDI).
- **95% confidence interval**: an interval *estimator* with long-run coverage. After seeing the data, it is not in general valid to say "there is 95% probability the parameter is in this interval".

A credible interval is not a "Bayesian confidence interval". Coverage is a frequentist property; posterior probability is a Bayesian one. They can numerically agree (e.g. flat-prior binomial with a lot of data) and can disagree (strong priors, small \(n\), discrete parameters).


## 2. Bernoulli / Binomial / Beta A/B — no MCMC required

**Generative story**

- One trial (shot, click, conversion): \(y_i \sim \mathrm{Bernoulli}(p)\).
- \(n\) i.i.d. trials, \(s=\sum y_i\) successes: \(s \sim \mathrm{Binomial}(n,p)\).
- Prior on the unknown rate: \(p \sim \mathrm{Beta}(\alpha,\beta)\).

**Why Beta?** It is supported on \((0,1)\) and **conjugate** to Bernoulli/Binomial: the posterior is the same family.

\[
p \mid s,n \;\sim\; \mathrm{Beta}(\alpha + s,\; \beta + n - s)
\]

- \(\mathrm{Beta}(1,1)\) is Uniform\((0,1)\): "I have no preference among rates before seeing data".
- \(\mathrm{Beta}(1,1)\) is *weak*, not "uninformative" in every decision sense — it still puts mass near 0 and 1.

The update is a counting argument: each success increments \(\alpha\), each failure increments \(\beta\). **No Metropolis, no NUTS.** Draw posterior samples with `scipy.stats.beta.rvs`.

Synthetic football-flavoured A/B: two similar set-piece routines, \(n=80\) each.


In [2]:
# Routine A vs B. Synthetic; i.i.d. Bernoulli is the *model*, not a claim about real set pieces.
n_a, s_a = 80, 8  # 10.0%
n_b, s_b = 80, 15  # 18.75%
alpha0, beta0 = 1, 1  # Uniform prior

# Analytical posteriors
post_a = (alpha0 + s_a, beta0 + n_a - s_a)  # Beta(9, 73)
post_b = (alpha0 + s_b, beta0 + n_b - s_b)  # Beta(16, 66)
print(f"posterior A: Beta{post_a}")
print(f"posterior B: Beta{post_b}")
print(f"posterior means: A={post_a[0] / sum(post_a):.3f}  B={post_b[0] / sum(post_b):.3f}")

rng = np.random.default_rng(SEED)
n_draw = 20_000
p_a = stats.beta.rvs(*post_a, size=n_draw, random_state=rng)
p_b = stats.beta.rvs(*post_b, size=n_draw, random_state=rng)
lift = p_b - p_a
threshold = 0.03  # practically meaningful: +3 percentage points

prob_better = float((p_b > p_a).mean())
ci = np.quantile(lift, [0.025, 0.975])
prob_material = float((lift > threshold).mean())

print(f"P(p_B > p_A | data)     = {prob_better:.3f}")
print(f"E[lift | data]          = {lift.mean():.3f}")
print(f"95% equal-tailed CI     = [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"P(lift > {threshold:.2f} | data)   = {prob_material:.3f}")

fig, axes = plt.subplots(1, 2)
grid = np.linspace(0.0, 0.45, 400)
axes[0].plot(grid, stats.beta.pdf(grid, *post_a), label="A")
axes[0].plot(grid, stats.beta.pdf(grid, *post_b), label="B")
axes[0].set_xlabel("conversion rate p")
axes[0].set_title("Beta posteriors (closed form)")
axes[0].legend()
axes[1].hist(lift, bins=50, density=True, color="0.35")
axes[1].axvline(0, color="C3", ls="--", lw=1)
axes[1].axvline(ci[0], color="k", ls=":", lw=1)
axes[1].axvline(ci[1], color="k", ls=":", lw=1)
axes[1].set_xlabel(r"$p_B - p_A$")
axes[1].set_title("posterior of lift (MC from Beta, not MCMC)")
plt.tight_layout()

posterior A: Beta(9, 73)
posterior B: Beta(16, 66)
posterior means: A=0.110  B=0.195
P(p_B > p_A | data)     = 0.941
E[lift | data]          = 0.085
95% equal-tailed CI     = [-0.023, 0.196]
P(lift > 0.03 | data)   = 0.846


/tmp/ipykernel_39317/1743752118.py:42: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


In [3]:
table = np.array([[s_a, n_a - s_a], [s_b, n_b - s_b]])
_, fisher_p = stats.fisher_exact(table, alternative="two-sided")
zstat, z_p = proportions_ztest([s_b, s_a], [n_b, n_a], alternative="two-sided")
print("Classical two-sided tests (H0: p_A = p_B)")
print(f"  Fisher exact p = {fisher_p:.3f}")
print(f"  two-proportion z = {zstat:.2f}, p = {z_p:.3f}")
print()
print("These answer different questions:")
print("  frequentist p-value:  P(data as or more extreme | H0 of equal rates)")
print("  Bayesian P(p_B>p_A):  P(B is better | data, prior, Bernoulli model)")
print("A small p-value is not P(H0 is false). A large P(p_B>p_A) is not a licence to ship")
print("if the lift CI still includes values too small to care about.")

Classical two-sided tests (H0: p_A = p_B)
  Fisher exact p = 0.175
  two-proportion z = 1.58, p = 0.115

These answer different questions:
  frequentist p-value:  P(data as or more extreme | H0 of equal rates)
  Bayesian P(p_B>p_A):  P(B is better | data, prior, Bernoulli model)
A small p-value is not P(H0 is false). A large P(p_B>p_A) is not a licence to ship
if the lift CI still includes values too small to care about.


**Beta draws vs MCMC.** Here the posterior *is* a named distribution, so `beta.rvs` is
exact (up to ordinary Monte Carlo error). MCMC is for posteriors without a convenient
form: products of non-conjugate pieces, hierarchies, logits of linear predictors.

**Do not claim a winner.** Use the p-value if the decision is "is there evidence against
equality under this design?". Use \(P(p_B>p_A\mid y)\) and the lift posterior if the
decision is "how sure are we that B is better, and by how much?". Report both when
stakeholders will otherwise confuse them. Independence of shots is assumed above;
real set pieces are clustered by match and taker.


## 3. Important distributions (interview pocket table)

| Distribution | Support | Typical role | Football / A-B example |
|---|---|---|---|
| **Bernoulli** | \(\{0,1\}\) | Binary outcome | Goal / no goal on one shot; convert / not |
| **Binomial** | \(0,\ldots,n\) | \(n\) i.i.d. Bernoulli counts | Goals in \(n\) similar penalties |
| **Beta** | \((0,1)\) | Prior/posterior for a probability | Prior on a conversion rate |
| **Normal** | \(\mathbb{R}\) | Coefficients, random effects, approx. likelihood | Player intercepts; \(\beta_{\text{distance}}\) |
| **HalfNormal** | \((0,\infty)\) | Scale prior (positive) | \(\sigma_{\text{player}}\) |
| **Student-\(t\)** | \(\mathbb{R}\) | Robust likelihood or heavier-tailed prior | Outlier-robust finishing residual; weakly informative \(\beta\) |
| **Poisson** | \(0,1,2,\ldots\) | Counts | Shots in a match; corners won |
| **Gamma** | \((0,\infty)\) | Positive continuous; Poisson rate prior | Prior on expected shots per match |
| **Dirichlet** | simplex \(\sum p_k=1\) | Composition / multinomial probabilities | Share of body parts; shot-type mix |

**Conjugacy** means the posterior stays in the same family as the prior, so the update is
arithmetic rather than sampling:

- Bernoulli / Binomial + **Beta** → Beta
- Poisson + **Gamma** → Gamma
- Categorical / Multinomial + **Dirichlet** → Dirichlet

Conjugate priors are computational conveniences, not scientific defaults. A conjugate
choice can still be a bad prior (too tight, wrong support, ignoring hierarchy).


## 4. Why MCMC — a football hierarchical sketch

From `01_data_exploration.ipynb`: **651 players, 2,918 non-shootout shots, median 3 shots
per player.** Raw conversion for a player with 3 shots is almost a coin flip.

A closed-form Beta–Binomial treats every shot as exchangeable with one global \(p\), or
gives each player their own \(p_j\) with no sharing. Finishing ability is *neither*.

Conceptual model (not fitted at full scale here):

\[
\text{goal}_{ij} \sim \mathrm{Bernoulli}(p_{ij})
\]

\[
\mathrm{logit}(p_{ij})
=
\beta_0
+ \beta_{\text{distance}}\,\text{distance}_{ij}
+ u_{\text{player}[i]}
\]

\[
u_j \sim \mathrm{Normal}(0, \sigma_{\text{player}}),
\qquad
\sigma_{\text{player}} \sim \mathrm{HalfNormal}(\cdot)
\]

The likelihood is Bernoulli-on-the-logit-scale; the prior on \(u_j\) is Gaussian; \(\sigma\)
is unknown. The posterior is **not** Beta. That product has no useful closed form, so we
sample it with MCMC.

| Pooling | What it does | Failure mode |
|---|---|---|
| **Complete** | One \(p\) for everyone | Star strikers and full-backs share a finishing rate |
| **No pooling** | Separate \(p_j\), no sharing | 3/3 looks like a 100% finisher |
| **Partial** | \(u_j\) drawn from a shared \(\mathrm{Normal}(0,\sigma)\) | Players with little data shrink toward the population; players with lots of data stay near their own rate |

**Shrinkage** is the pull of \(u_j\) toward 0 (the group mean on the logit scale). It is
strongest when \(n_j\) is small or \(\sigma_{\text{player}}\) is estimated small. It is a
feature: it encodes "extraordinary rates need extraordinary evidence".


## 5. Small actual PyMC model

Twelve open-play shooters from the canonical StatsBomb subset: six high-volume, six
low-volume (including Yerry Mina 3/3). Objective: syntax and shrinkage, not a production
xG model. Distance is centered; angle and body part are omitted on purpose.


In [4]:
shots = read_shot_dataset()
open_play = shots.loc[shots["shot_type"] == "Open Play"].copy()

# High volume + sparse extremes (deterministic IDs from the development extract).
PLAYER_IDS = [
    5474,
    10955,
    3289,
    4320,
    5625,
    3233,  # Perišić, Kane, Lukaku, Neymar, Forsberg, Sterling
    6196,
    5473,
    11601,
    3166,
    3318,
    3382,  # Mina 3/3, Musa 2/4, Pessina 2/4, Verratti 0/4, Rashford 0/4, Shaw 1/3
]
subset = open_play.loc[open_play["player_id"].isin(PLAYER_IDS)].copy()
subset["distance_c"] = subset["shot_distance"] - subset["shot_distance"].mean()
player_idx, player_names = pd.factorize(subset["player"], sort=True)

raw = (
    subset.groupby("player", sort=True)
    .agg(shots=("goal", "size"), goals=("goal", "sum"))
    .assign(raw_rate=lambda d: d["goals"] / d["shots"])
)
print(f"shots={len(subset)}  players={len(player_names)}  goal rate={subset['goal'].mean():.3f}")
print(raw.to_string())

coords = {"player": list(player_names)}
with pm.Model(coords=coords) as finishing:
    intercept = pm.Normal("intercept", mu=-2.2, sigma=1.0)  # logit(0.1) ≈ -2.2
    beta_distance = pm.Normal("beta_distance", mu=0.0, sigma=0.1)  # per yard, centered
    sigma_player = pm.HalfNormal("sigma_player", sigma=0.5)
    z = pm.Normal("z", 0.0, 1.0, dims="player")  # non-centered
    player_effect = pm.Deterministic("player_effect", z * sigma_player, dims="player")

    logit_p = (
        intercept + beta_distance * subset["distance_c"].to_numpy() + player_effect[player_idx]
    )
    pm.Bernoulli("goal", logit_p=logit_p, observed=subset["goal"].to_numpy().astype(int))

print(finishing)
print("free RVs:", [rv.name for rv in finishing.free_RVs])

shots=176  players=12  goal rate=0.205
                               shots  goals  raw_rate
player                                               
Ahmed Musa                         4      2  0.500000
Emil Peter Forsberg               25      4  0.160000
Harry Kane                        26      7  0.269231
Ivan Perišić                      29      5  0.172414
Luke Shaw                          3      1  0.333333
Marco Verratti                     4      0  0.000000
Marcus Rashford                    4      0  0.000000
Matteo Pessina                     4      2  0.500000
Neymar da Silva Santos Junior     25      2  0.080000
Raheem Sterling                   24      3  0.125000
Romelu Lukaku Menama              25      7  0.280000
Yerry Fernando Mina González       3      3  1.000000


free RVs: ['intercept', 'beta_distance', 'sigma_player', 'z']


In [5]:
with finishing:
    prior_idata = pm.sample_prior_predictive(draws=400, random_seed=SEED)
    idata = pm.sample(
        draws=500,
        tune=500,
        chains=4,
        cores=1,
        target_accept=0.9,
        random_seed=SEED,
        progressbar=True,
    )
    pm.sample_posterior_predictive(idata, extend_inferencedata=True, random_seed=SEED)

idata.extend(prior_idata)
print(az.summary(idata, var_names=["intercept", "beta_distance", "sigma_player"], round_to=3))

Sampling: [beta_distance, goal, intercept, sigma_player, z]


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (4 chains in 1 job)


NUTS: [intercept, beta_distance, sigma_player, z]


/home/tom/projects/football-intelligence-lab/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 4 seconds.


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


Sampling: [goal]


/home/tom/projects/football-intelligence-lab/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

                mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
intercept     -1.715  0.273  -2.231   -1.205      0.006    0.006  2079.326   
beta_distance -0.141  0.034  -0.207   -0.083      0.001    0.001  2836.093   
sigma_player   0.344  0.247   0.000    0.791      0.007    0.004   963.509   

               ess_tail  r_hat  
intercept      1240.106  1.000  
beta_distance  1435.006  1.001  
sigma_player   1030.601  1.001  


## 6. MCMC algorithms (what to say out loud)

**Metropolis–Hastings.** Propose \(\theta' \sim q(\theta'\mid \theta)\). Accept with

\[
\alpha = \min\left(1,\;
\frac{p(\theta'\mid y)\,q(\theta\mid\theta')}{p(\theta\mid y)\,q(\theta'\mid\theta)}
\right).
\]

Random-walk MH takes tiny correlated steps in high dimensions: most proposals are
implausible, acceptance collapses, chains mix slowly. You *can* use it; you rarely
*should* for continuous hierarchical models.

**Hamiltonian Monte Carlo (HMC).** Treat \(-\log p(\theta\mid y)\) as a potential energy
surface. Introduce momentum and simulate a trajectory that uses **gradients of the log
posterior**. The chain can travel far in one accepted proposal, so exploration of
continuous high-dimensional posteriors is much more efficient than random walk.

**NUTS (No-U-Turn Sampler).** Adaptive HMC: grow the trajectory until it would double
back ("U-turn"), so you do not have to hand-tune the number of leapfrog steps. PyMC's
default for suitable continuous parameters. Discrete parameters need other kernels.

Interview stop: "MH is accept/reject random walk; HMC uses gradients to follow the
posterior geometry; NUTS is HMC that adapts trajectory length. PyMC uses NUTS here."


## 7. Diagnostics — sampling quality, not model quality


In [6]:
rhat = az.rhat(idata)
ess = az.ess(idata)
n_div = int(idata.sample_stats["diverging"].sum())
rhat_max = {name: float(values.max()) for name, values in rhat.items()}
worst = max(rhat_max, key=rhat_max.get)
print(f"max R-hat          = {rhat_max[worst]:.4f}  ({worst})")
print(f"min ESS (bulk)     = {float(ess.to_array().min()):.0f}")
print(f"divergences        = {n_div}   (0 is desirable; not a proof the model is right)")
print("R-hat by group:", {k: round(v, 4) for k, v in rhat_max.items()})

az.plot_trace(
    idata,
    var_names=["intercept", "beta_distance", "sigma_player"],
    compact=True,
)

max R-hat          = 1.0116  (z)
min ESS (bulk)     = 964
divergences        = 0   (0 is desirable; not a proof the model is right)
R-hat by group: {'intercept': 1.0003, 'beta_distance': 1.0008, 'z': 1.0116, 'sigma_player': 1.0009, 'player_effect': 1.0047}


array([[<Axes: title={'center': 'intercept'}>,
        <Axes: title={'center': 'intercept'}>],
       [<Axes: title={'center': 'beta_distance'}>,
        <Axes: title={'center': 'beta_distance'}>],
       [<Axes: title={'center': 'sigma_player'}>,
        <Axes: title={'center': 'sigma_player'}>]], dtype=object)

In [7]:
# Shrinkage: raw conversion vs posterior mean rate at average distance (distance_c = 0).
post_rate = expit(idata.posterior["intercept"] + idata.posterior["player_effect"]).mean(
    ("chain", "draw")
)
compare = raw.copy()
compare["post_rate"] = post_rate.to_pandas().reindex(compare.index)
compare["shrink"] = compare["post_rate"] - compare["raw_rate"]
print(compare.sort_values("shots").to_string())

fig, ax = plt.subplots()
sizes = 30 + 8 * compare["shots"].to_numpy()
ax.scatter(compare["raw_rate"], compare["post_rate"], s=sizes, c="C0", alpha=0.85)
for name, row in compare.iterrows():
    label = str(name).split(" ")[-1]
    ax.annotate(
        label,
        (row["raw_rate"], row["post_rate"]),
        fontsize=8,
        xytext=(4, 2),
        textcoords="offset points",
    )
lims = [0, max(0.35, float(compare[["raw_rate", "post_rate"]].max().max()) + 0.05)]
ax.plot(lims, lims, color="0.5", ls="--", lw=1)
ax.set_xlabel("raw conversion")
ax.set_ylabel("posterior mean rate at mean distance")
ax.set_title("Partial pooling: 3/3 shrinks; 7/26 barely moves (marker size = shots)")

                               shots  goals  raw_rate  post_rate    shrink
player                                                                    
Luke Shaw                          3      1  0.333333   0.175067 -0.158266
Yerry Fernando Mina González       3      3  1.000000   0.196431 -0.803569
Marco Verratti                     4      0  0.000000   0.149931  0.149931
Ahmed Musa                         4      2  0.500000   0.185101 -0.314899
Matteo Pessina                     4      2  0.500000   0.176585 -0.323415
Marcus Rashford                    4      0  0.000000   0.155536  0.155536
Raheem Sterling                   24      3  0.125000   0.134203  0.009203
Emil Peter Forsberg               25      4  0.160000   0.172829  0.012829
Neymar da Silva Santos Junior     25      2  0.080000   0.127382  0.047382
Romelu Lukaku Menama              25      7  0.280000   0.162435 -0.117565
Harry Kane                        26      7  0.269231   0.164009 -0.105222
Ivan Perišić             

Text(0.5, 1.0, 'Partial pooling: 3/3 shrinks; 7/26 barely moves (marker size = shots)')

**R-hat** (Gelman–Rubin / rank-normalized): compares within-chain and between-chain
variation. Values \(\approx 1\) mean the chains have forgotten their starts and agree.
\(\hat R \gtrsim 1.01\) is a red flag, not a soft suggestion.

**ESS** (effective sample size): MCMC draws are autocorrelated, so 1,000 draws are not
1,000 independent facts. Bulk ESS for means, tail ESS for interval endpoints.

**Trace plots**: healthy chains overlap in a stable band ("hairy caterpillar"). Trends,
stuck chains, or chain-specific plateaus mean do not trust the summary yet.

**Divergences**: NUTS could not simulate the Hamiltonian trajectory accurately —
often a geometry problem (funnel), not a "software error". Zero divergences \(\neq\)
correct model; many divergences \(\neq\) "just run more draws".

**Funnel / non-centered parameterisation.** Centered form \(u_j \sim N(0,\sigma)\)
couples \(u_j\) and \(\sigma\): when \(\sigma\) is near 0, the \(u_j\) must sit in a
narrow spike (Neal's funnel). HMC diverges. Non-centered:

\[
z_j \sim \mathrm{Normal}(0,1), \qquad u_j = z_j \cdot \sigma_{\text{player}}
\]

We used that above. It is not always better (lots of data per group can favour centered),
but it is the default trick to mention for hierarchical scales.


## 8. Predictive checks

- **Prior predictive:** simulate data from the *prior* only. Question: *before seeing
  observations, does this prior imply plausible conversion rates?* A prior that puts 80%
  of shots as goals is a bad finishing prior, even if NUTS will run.
- **Posterior predictive:** simulate data from the *fitted* posterior. Question: *can the
  model generate datasets that resemble what we saw?*

Passing R-hat / ESS / zero divergences means **the sampler worked**. It does not mean the
Bernoulli-logit-plus-distance story is a good model of shooting.


In [8]:
goal_prior = prior_idata.prior_predictive["goal"]
goal_post = idata.posterior_predictive["goal"]
prior_rate = goal_prior.mean([d for d in goal_prior.dims if d not in ("chain", "draw")])
post_rate_rep = goal_post.mean([d for d in goal_post.dims if d not in ("chain", "draw")])
obs = float(subset["goal"].mean())

print(f"observed conversion              = {obs:.3f}")
print(f"prior predictive rate 95%        = {np.quantile(np.asarray(prior_rate), [0.025, 0.975])}")
print(
    f"posterior predictive rate 95%    = {np.quantile(np.asarray(post_rate_rep), [0.025, 0.975])}"
)

fig, axes = plt.subplots(1, 2)
axes[0].hist(np.asarray(prior_rate).ravel(), bins=30, density=True, color="C0", alpha=0.85)
axes[0].axvline(obs, color="C3", ls="--")
axes[0].set_title("prior predictive conversion")
axes[0].set_xlabel("mean(y)")
axes[1].hist(np.asarray(post_rate_rep).ravel(), bins=30, density=True, color="C1", alpha=0.85)
axes[1].axvline(obs, color="C3", ls="--", label="observed")
axes[1].set_title("posterior predictive conversion")
axes[1].set_xlabel("mean(y)")
axes[1].legend()
print("The red line is the observed rate. PPC matching the mean is necessary, not sufficient:")
print("we did not check calibration by distance, clustering by match, or leftover overdispersion.")

observed conversion              = 0.205
prior predictive rate 95%        = [0.01704545 0.47173295]
posterior predictive rate 95%    = [0.13068182 0.27840909]
The red line is the observed rate. PPC matching the mean is necessary, not sufficient:
we did not check calibration by distance, clustering by match, or leftover overdispersion.


## 9. 60-second Bayesian answers

**Why Bayesian inference?**  
To put a probability distribution on unknowns, fold prior information / regularisation
in explicitly, and answer questions like \(P(\theta > 0 \mid y)\) or the distribution of
a decision-relevant functional (lift, player effect). Not because p-values are "wrong".

**What is a conjugate prior?**  
A prior that keeps the posterior in the same family (Beta–Binomial, Gamma–Poisson,
Dirichlet–Multinomial). Convenient, not automatically wise.

**When do we need MCMC?**  
When the posterior has no usable closed form: non-conjugate likelihoods, logits,
hierarchies, unknown scales. If it *is* Beta, just use Beta.

**HMC vs Metropolis?**  
Metropolis: random-walk propose/accept, slow in high dimension. HMC: uses gradients of
the log posterior to travel along energy trajectories, much better mixing for continuous
parameters.

**What is NUTS?**  
No-U-Turn Sampler: HMC that grows trajectories until they would U-turn, so leapfrog
length is not a hand-tuned knob. PyMC default for continuous variables.

**R-hat and ESS?**  
R-hat \(\approx 1\): chains agree. ESS: independent-equivalent sample size after
autocorrelation. Diagnostics of *sampling*, not of scientific truth.

**What is a divergence?**  
A warning that the HMC trajectory was numerically untrustworthy, often hierarchical
funnel geometry. Fix the parameterisation / priors; do not silently discard divergences.

**Why hierarchical models for football?**  
Shots nest in players (and matches, teams). Sample sizes are wildly unbalanced (median
3 shots). Ignoring that overstates certainty for noisy players.

**What is partial pooling?**  
Share strength through a common distribution of player effects. Low-\(n\) players shrink
toward the group; high-\(n\) players keep their signal. Between complete pooling and
no pooling.

**Credible interval vs confidence interval?**  
Credible: \(P(\theta \in I \mid y)=0.95\) under the posterior. Confidence: long-run
coverage of a procedure. Do not swap the interpretations.

**Prior predictive vs posterior predictive?**  
Prior: are the priors generating sane data? Posterior: can the fitted model reproduce
the observations? Both can fail while R-hat looks perfect.

**Bayesian A/B: what would you report?**  
The posterior of each rate, \(P(p_B>p_A\mid y)\), the posterior of lift, a 95% credible
interval, and \(P(\text{lift} > \delta)\) for a pre-stated practical \(\delta\). Mention
the prior and the i.i.d. assumption. Optionally place a classical test beside it and
say the questions differ.

### What this notebook did *not* do

No production hierarchical finishing module, no full-roster MCMC, no causal claim that
a player "is better", no treating SHAP or p-values as posterior probabilities.
